This notebook aims to generate relevant graphs for visualisation concerns 

In [1]:
import matplotlib.pyplot as plt 
import numpy as np 

import torch

import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset 


import torchvision 
import torchvision.transforms as transforms 

from sklearn.cluster import KMeans 
from sklearn.neighbors import NearestNeighbors

from typing import List, Tuple 


In [2]:
def get_initial_seed(train_dataset, n_per_class=1):
    labels = np.array(train_dataset.targets)
    seed = []
    for c in range(10): 
        idx = np.where(labels == c)[0]
        seed.append(int(np.random.choice(idx)))
    return seed

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

base_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        (0.4914, 0.4822, 0.4465),
        (0.2470, 0.2435, 0.2616)
    ),
])

train_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=base_transform
)

test_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=base_transform
)

test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)
budgets = [10, 20, 40, 80]

class SimpleClassifier(nn.Module):
    def __init__(self, dropout=False):
        super().__init__()
        self.dropout = dropout

        self.features = torchvision.models.resnet18(pretrained=False)
        self.features.fc = nn.Identity()  # remove final layer

        self.classifier = nn.Sequential(
            nn.Dropout(0.5) if dropout else nn.Identity(),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        feat = self.features(x)
        return self.classifier(feat)


/home/ariag/5CCSAMLF/large_meshupar/env/lib/python3.12/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [4]:
def score_least_confidence(logits):
    probs = torch.softmax(logits ,  dim=1)
    return ( 1- probs.max(dim=1).values)

In [5]:
def score_margin(logits):
    probs = torch.softmax(logits,dim=1)
    top2 = torch.topk(probs, 2, dim=1).values

    return top2[:,0] - top2[:,1] 


In [6]:
def score_entropy(logits):
    probs = torch.softmax(logits,dim=1)
    return - 1 * ( probs * probs.log()).sum(dim=1)

In [7]:
def compute_badge_embeddings(model, loader, device):
    model.eval()
    grads = [] 



    for x, _ in loader:
        x = x.to(device)
        x.requires_grad=True

        logits =model(x)
        probs = torch.softmax(logits, dim=1)

        y_hat = probs.argmax(dim=1)



        loss = F.cross_entropy(logits, y_hat, reduction='sum')
        loss.backward()


        grads.append(x.grad.view(x.size(0), -1).cpu().numpy())
    return np.concatenate(grads, axis=0)

In [8]:
def select_by_uncertainty(model, unlabeled_loader, device, B, scoring_fn):
    model.eval()
    scores = []

    with torch.no_grad():
        for x, idx in unlabeled_loader:
            x=x.to(device)
            logits = model(x)
            s =  scoring_fn(logits)
            scores.extend(list(zip(idx.numpy(), s.cpu(). numpy()) ))

    # desc. uncertainity
    scores = sorted(scores, key=lambda x: -x[1])
    selected = [idx for idx, _ in scores[:B]]

    return selected 

In [9]:
def plot_baseline_comparison(results_dict, budgets):
    plt.figure(figsize=(8,5))
    for method, accs in results_dict.items():
        plt.plot(budgets, accs, marker="x", label=method)

    plt.xlabel("Budget")
    plt.ylabel("Accuracy")

    plt.title("Active Learning baseline comparison")

    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()

Active Learning Loop Section 

In [10]:
def evaluate(model, loader, device): 
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in loader:
            x,y = x.to(device), y.to(device)
            logits = model(x)

            preds = logits.argmax(dim=1)
            correct += (preds ==y ).sum().item()

            total += y.size(0)
        
        return correct / total 

In [11]:
def active_learning_round(
    model,
    train_dataset,
    test_loader,
    labeled_indices,
    unlabeled_indices,
    B,
    scoring_fn,
    device
):
    labeled_subset = torch.utils.data.Subset(train_dataset, labeled_indices)
    loader = DataLoader(labeled_subset, batch_size=128, shuffle=True)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()

    model.train()
    for epoch in range(5):  
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = loss_fn(logits, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    unlabeled_subset = torch.utils.data.Subset(train_dataset, unlabeled_indices)
    unlabeled_loader = DataLoader(unlabeled_subset, batch_size=128, shuffle=False)

    selected = select_by_uncertainty(model, unlabeled_loader, device, B, scoring_fn)
    selected_global = [unlabeled_indices[i] for i in selected]

    new_labeled = labeled_indices + selected_global
    new_unlabeled = [i for i in unlabeled_indices if i not in selected_global]

    acc = evaluate(model, test_loader, device)

    return new_labeled, new_unlabeled, acc


In [12]:
def mc_dropout_predict(model, x, T=3):
    model.train()  # keep dropout ON
    preds = []
    for _ in range(T):
        logits = model(x)
        preds.append(torch.softmax(logits, dim=1).unsqueeze(0))
    return torch.cat(preds, dim=0)  # [t, b,   c]

# BALD method 
def score_bald_from_mc(mc_probs):
    mean_probs = mc_probs.mean(dim=0)
    entropy_mean = -(mean_probs * mean_probs.log()).sum(dim=1)
    entropy_expected = -(mc_probs * mc_probs.log()).sum(dim=2).mean(dim=0)
    return entropy_mean - entropy_expected


def score_bald(model, x, device, T=3):
    x = x.to(device)
    mc_probs = mc_dropout_predict(model, x, T=T)
    return score_bald_from_mc(mc_probs)

def score_dbal_from_mc(mc_probs):
    preds = mc_probs.argmax(dim=2)  # [T, B]
    mode_counts = torch.mode(preds, dim=0).values
    return 1 - (mode_counts / mc_probs.size(0))


In [13]:
def select_badge(model, unlabeled_loader, device, B):
    model.eval()
    embeddings = compute_badge_embeddings(model, unlabeled_loader, device)

    # kmeans method 
    kmeans = KMeans(n_clusters=B, init='k-means++')
    kmeans.fit(embeddings)

    centers = kmeans.cluster_centers_
    dists = np.linalg.norm(embeddings[:, None] - centers[None, :], axis=2)
    selected = np.argmin(dists, axis=0)

    all_indices = []
    for _, idx in unlabeled_loader:
        all_indices.extend(idx.numpy())

    return [all_indices[i] for i in selected]


In [14]:
def select_random(unlabeled_indices, B):
    return list(np.random.choice(unlabeled_indices, size=B, replace=False))


In [15]:
def evaluate_fixed_selection(model, train_dataset, test_loader, indices, device):
    subset = Subset(train_dataset, indices)
    loader = DataLoader(subset, batch_size=128, shuffle=True)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()

    model.train()
    for epoch in range(5):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = loss_fn(logits, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    return evaluate(model, test_loader, device)


In [16]:
def run_all_baselines(train_dataset, test_loader, budgets, device):
    print("\n=== Starting all baselines ===")

    results = {
        "Least Confidence": [],
        "Margin": [],
        "Entropy": [],
        "BADGE": [],
        "BALD": [],
        "DBAL": [],
        "Random": [],
        "TPC-RP": [], 
        "TPC-DC": []  
    }

    scoring_fns = {
        "Least Confidence": score_least_confidence,
        "Margin": score_margin,
        "Entropy": score_entropy
    }


    for method, fn in scoring_fns.items():
        print(f"\n=== Running baseline: {method} ===")
        labeled = get_initial_seed(train_dataset)
        unlabeled = list(range(len(train_dataset)))
        accs = []

        for B in budgets:
            print(f"  -> Budget {B}: training classifier...")
            model = SimpleClassifier().to(device)

            labeled, unlabeled, acc = active_learning_round(
                model, train_dataset, test_loader,
                labeled, unlabeled, B,
                fn, device
            )

            print(f"     Accuracy after budget {B}: {acc:.4f}")
            accs.append(acc)

        results[method] = accs


    print("\n=== Running baseline: BADGE ===")
    labeled = get_initial_seed(train_dataset)
    unlabeled = list(range(len(train_dataset)))
    accs = []

    for B in budgets:
        print(f"  -> BADGE selecting {B} points...")
        model = SimpleClassifier().to(device)

        unlabeled_subset = Subset(train_dataset, unlabeled)
        unlabeled_loader = DataLoader(unlabeled_subset, batch_size=128, shuffle=False)

        selected = select_badge(model, unlabeled_loader, device, B)
        labeled += selected
        unlabeled = [i for i in unlabeled if i not in selected]

        acc = evaluate(model, test_loader, device)
        print(f"     Accuracy after BADGE budget {B}: {acc:.4f}")
        accs.append(acc)

    results["BADGE"] = accs


    print("\n=== Running baseline: BALD ===")
    labeled = get_initial_seed(train_dataset)
    unlabeled = list(range(len(train_dataset)))
    accs = []

    for B in budgets:
        print(f"  -> BALD scoring unlabeled pool for budget {B}...")
        model = SimpleClassifier(dropout=True).to(device)

        unlabeled_subset = Subset(train_dataset, unlabeled)
        unlabeled_loader = DataLoader(unlabeled_subset, batch_size=128, shuffle=False)

        selected = []
        for x, idx in unlabeled_loader:
            scores = score_bald(model, x, device)
            selected.extend(list(zip(idx.numpy(), scores.cpu().numpy())))

        selected = sorted(selected, key=lambda x: -x[1])[:B]
        selected = [i for i, _ in selected]

        labeled += selected
        unlabeled = [i for i in unlabeled if i not in selected]

        acc = evaluate(model, test_loader, device)
        print(f"     Accuracy after BALD budget {B}: {acc:.4f}")
        accs.append(acc)

    results["BALD"] = accs



    print("\n=== Running baseline: DBAL ===")
    labeled = get_initial_seed(train_dataset)
    unlabeled = list(range(len(train_dataset)))
    accs = []

    for B in budgets:
        print(f"  -> DBAL scoring unlabeled pool for budget {B}...")
        model = SimpleClassifier(dropout=True).to(device)

        unlabeled_subset = Subset(train_dataset, unlabeled)
        unlabeled_loader = DataLoader(unlabeled_subset, batch_size=128, shuffle=False)

        selected = []
        for x, idx in unlabeled_loader:
            mc_probs = mc_dropout_predict(model, x.to(device))
            scores = score_dbal_from_mc(mc_probs)
            selected.extend(list(zip(idx.numpy(), scores.cpu().numpy())))

        selected = sorted(selected, key=lambda x: -x[1])[:B]
        selected = [i for i, _ in selected]

        labeled += selected
        unlabeled = [i for i in unlabeled if i not in selected]

        acc = evaluate(model, test_loader, device)
        print(f"     Accuracy after DBAL budget {B}: {acc:.4f}")
        accs.append(acc)

    results["DBAL"] = accs

    print("\n=== Evaluating TPC-RP selections ===")
    for B in budgets:
        print(f"  -> Evaluating TPC-RP for budget {B}...")
        indices = np.load(f"typiclust_B{B}.npy")
        model = SimpleClassifier().to(device)
        acc = evaluate_fixed_selection(model, train_dataset, test_loader, indices, device)
        print(f"     TPC-RP accuracy for budget {B}: {acc:.4f}")
        results["TPC-RP"].append(acc)
    print("\n=== Evaluating TPC-DC selections ===")
    for B in budgets:
        print(f"  -> Evaluating TPC-DC for budget {B}...")
        try:
            indices = np.load(f"tpcdc_B{B}.npy")
            model = SimpleClassifier().to(device)
            acc = evaluate_fixed_selection(model, train_dataset, test_loader, indices, device)
            print(f"     TPC-DC accuracy for budget {B}: {acc:.4f}")
            results["TPC-DC"].append(acc)
        except FileNotFoundError:
            print(f"     No TPC-DC file found for budget {B}.")
            results["TPC-DC"].append(None)

    print("\n=== All baselines complete ===")
    return results


In [ ]:
results = run_all_baselines(train_dataset, test_loader, budgets, device)
plot_baseline_comparison(results, budgets)



=== Starting all baselines ===

=== Running baseline: Least Confidence ===
  -> Budget 10: training classifier...


/home/ariag/5CCSAMLF/large_meshupar/env/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/ariag/5CCSAMLF/large_meshupar/env/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


     Accuracy after budget 10: 0.1233
  -> Budget 20: training classifier...
     Accuracy after budget 20: 0.1498
  -> Budget 40: training classifier...
     Accuracy after budget 40: 0.1608
  -> Budget 80: training classifier...
     Accuracy after budget 80: 0.1457

=== Running baseline: Margin ===
  -> Budget 10: training classifier...
     Accuracy after budget 10: 0.1311
  -> Budget 20: training classifier...
     Accuracy after budget 20: 0.1249
  -> Budget 40: training classifier...
     Accuracy after budget 40: 0.1421
  -> Budget 80: training classifier...
     Accuracy after budget 80: 0.1695

=== Running baseline: Entropy ===
  -> Budget 10: training classifier...
     Accuracy after budget 10: 0.1216
  -> Budget 20: training classifier...
     Accuracy after budget 20: 0.1063
  -> Budget 40: training classifier...
     Accuracy after budget 40: 0.1556
  -> Budget 80: training classifier...
     Accuracy after budget 80: 0.1548

=== Running baseline: BADGE ===
  -> BADGE se

/home/ariag/5CCSAMLF/large_meshupar/env/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/ariag/5CCSAMLF/large_meshupar/env/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)
